# RepoEval — Experiment Result Figures & Tables

This notebook generates all figures and tables for the RepoEval benchmark results.

Data source: `eval/repoeval/result/{api,line}_all_results.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pathlib import Path

# Paths
RESULT_DIR = Path("../repoeval/result")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load data
api_df = pd.read_csv(RESULT_DIR / "api_all_results.csv")
api_df["split"] = "api"
line_df = pd.read_csv(RESULT_DIR / "line_all_results.csv")
line_df["split"] = "line"
df = pd.concat([api_df, line_df], ignore_index=True)

# Filter out starcoder2-7b
df = df[df["llm"] != "starcoder2-7b"]

# Separate baseline and RAG configurations
baseline = df[df["method"] == "baseline"].copy()
rag = df[df["method"] != "baseline"].copy()

print(f"Total rows: {len(df)}, RAG rows: {len(rag)}, Baseline rows: {len(baseline)}")
print(f"Methods: {sorted(rag['method'].unique())}")
print(f"Retrievers: {sorted(rag['retriever'].unique())}")
print(f"LLMs: {sorted(rag['llm'].unique())}")

## RQ1: Strategy Effect — Table

Mean EM per chunking method, averaged across all retriever × LLM × parameter configurations.
Export: CSV only. LaTeX tabular filled manually in the main tex file.

In [ ]:
# RQ1 — Mean EM per method across all configurations
# Export: output/repoeval_rq1_strategy_effect.csv

rq1 = df.groupby(["method", "split"])["EM"].agg(["mean", "std", "count"]).reset_index()
rq1.columns = ["method", "split", "EM", "std", "n"]
rq1 = rq1.sort_values(["split", "method"])
rq1.to_csv(OUTPUT_DIR / "repoeval_rq1_strategy_effect.csv", index=False)

print("Exported to output/repoeval_rq1_strategy_effect.csv")
print(rq1.to_string(index=False))

## RQ1: Strategy Effect — Statistical Tests

Pairwise Wilcoxon signed-rank + Cliff's delta across all 6 strategy pairs.
Each observation = EM of one (retriever × LLM × chunk_size × context_length) configuration (n=144 per strategy).
Correction: Holm–Bonferroni for 6 comparisons.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

METHOD_ORDER = ["function", "declaration", "sliding", "cast"]
METHOD_LABELS_MAP = {
    "function": "Function", "declaration": "Declaration",
    "sliding": "Sliding Window", "cast": "cAST"
}

def cliffs_delta(x, y):
    """Cliff's delta for paired samples."""
    n = len(x)
    more = sum(1 for i in range(n) if x[i] > y[i])
    less = sum(1 for i in range(n) if x[i] < y[i])
    return (more - less) / n

def interpret_delta(d):
    d = abs(d)
    if d < 0.147:   return "Negligible"
    elif d < 0.33:  return "Small"
    elif d < 0.474: return "Medium"
    else:           return "Large"

pairs = list(combinations(METHOD_ORDER, 2))
all_results = []

for split_name in ["api", "line"]:
    sub = rag[rag["split"] == split_name]
    pivot = sub.pivot_table(
        index=["retriever", "llm", "max_chunk_size", "max_crossfile_context"],
        columns="method", values="EM"
    )

    p_values = []
    results = []
    for a, b in pairs:
        x, y = pivot[a].values, pivot[b].values
        stat, p = wilcoxon(x, y)
        delta = cliffs_delta(x, y)
        p_values.append(p)
        results.append({
            "split": split_name,
            "strategy_A": METHOD_LABELS_MAP[a],
            "strategy_B": METHOD_LABELS_MAP[b],
            "wilcoxon_stat": stat,
            "p_raw": p,
            "cliffs_delta": round(delta, 4),
            "effect_size": interpret_delta(delta),
        })

    rejected, corrected_p, _, _ = multipletests(p_values, method="holm")
    for i, r in enumerate(results):
        r["p_corrected"] = corrected_p[i]
        r["significant"] = "Yes" if rejected[i] else "No"

    all_results.extend(results)

rq1_stat_df = pd.DataFrame(all_results)
rq1_stat_df = rq1_stat_df[["split", "strategy_A", "strategy_B", "wilcoxon_stat",
                    "p_raw", "p_corrected", "significant", "cliffs_delta", "effect_size"]]

rq1_stat_df.to_csv(OUTPUT_DIR / "repoeval_rq1_statistical_tests.csv", index=False)
print("Exported to output/repoeval_rq1_statistical_tests.csv\n")
print(rq1_stat_df.to_string(index=False))

## RQ2: Interaction Effect — Tables

Table (a): Method × Retriever — mean EM across LLM × params.
Table (b): Method × Generator — mean EM across retriever × params.
Bold best method per column to show if ranking is consistent.

In [ ]:
# RQ2(a) — Method × Retriever interaction table
# Mean EM across LLM × parameter settings, per split

RETRIEVER_ORDER = ["bm25", "embeddinggemma-300m", "Qwen3-Embedding-0.6B", "Qwen3-Embedding-4B"]

for split_name in ["api", "line"]:
    sub = rag[rag["split"] == split_name]
    pivot = sub.groupby(["method", "retriever"])["EM"].mean().unstack("retriever")
    pivot.columns.name = None
    pivot = pivot.reindex(index=["function", "declaration", "sliding", "cast"],
                          columns=RETRIEVER_ORDER)
    pivot.round(4).to_csv(OUTPUT_DIR / f"repoeval_rq2a_method_retriever_{split_name}.csv")
    print(f"\n=== RQ2(a) Method × Retriever — {split_name} split ===")
    print(pivot.round(4).to_string())

In [ ]:
# RQ2(b) — Method × Generator interaction table
# Mean EM across retriever × parameter settings, per split

LLM_ORDER = ["deepseek-coder-6.7b-base", "Qwen2.5-Coder-7B", "Qwen3.5-9B-Base", "Seed-Coder-8B-Base"]

for split_name in ["api", "line"]:
    sub = rag[rag["split"] == split_name]
    pivot = sub.groupby(["method", "llm"])["EM"].mean().unstack("llm")
    pivot.columns.name = None
    pivot = pivot.reindex(index=["function", "declaration", "sliding", "cast"],
                          columns=LLM_ORDER)
    pivot.round(4).to_csv(OUTPUT_DIR / f"repoeval_rq2b_method_generator_{split_name}.csv")
    print(f"\n=== RQ2(b) Method × Generator — {split_name} split ===")
    print(pivot.round(4).to_string())

## RQ3: Parameter Sensitivity — Heatmap + Line Chart

Figure (a): Heatmap — 4 subplots (one per strategy), rows = chunk size, cols = context length, cell = mean EM.
Figure (b): Line chart — 2 subplots: (left) EM vs chunk size, (right) EM vs context length, one line per strategy.
Both figures generated per split. Export as PDF.

In [ ]:
# RQ3 — Heatmap: chunk_size × cross-file context length per strategy (2×4 grid)
# Style adapted from Chunk-legacy/eval/repoeval/visualize_results.ipynb

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

METHOD_ORDER = ["function", "declaration", "sliding", "cast"]
METHOD_LABELS = ["Function", "Declaration", "Sliding Window", "cAST"]
SPLIT_LABELS = {"api": "API-Level", "line": "Line-Level"}
CHUNK_SIZES = [1000, 2000, 3000]
CONTEXT_LENS = [2048, 4096, 8192]
METRIC = "EM"

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.linewidth': 0.6,
})

# Pre-compute all grids
grids = {}
all_vals = {s: [] for s in ["api", "line"]}
for split_name in ["api", "line"]:
    sub = rag[rag["split"] == split_name]
    for method in METHOD_ORDER:
        m = sub[sub["method"] == method]
        grid = np.zeros((len(CHUNK_SIZES), len(CONTEXT_LENS)))
        for r, cs in enumerate(CHUNK_SIZES):
            for c, cl in enumerate(CONTEXT_LENS):
                vals = m[(m["max_chunk_size"] == cs) & (m["max_crossfile_context"] == cl)][METRIC].values
                grid[r, c] = np.mean(vals) if len(vals) > 0 else 0
                all_vals[split_name].append(grid[r, c])
        grids[(split_name, method)] = grid

fig, axes = plt.subplots(2, 4, figsize=(7, 3.5), constrained_layout=True)

for row_idx, split_name in enumerate(["api", "line"]):
    vmin = np.floor(min(all_vals[split_name]) * 100) / 100
    vmax = np.ceil(max(all_vals[split_name]) * 100) / 100
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.cm.YlOrRd

    for col_idx, (method, mlabel) in enumerate(zip(METHOD_ORDER, METHOD_LABELS)):
        ax = axes[row_idx, col_idx]
        grid = grids[(split_name, method)]

        im = ax.imshow(grid, cmap=cmap, norm=norm, aspect="auto")

        for r in range(len(CHUNK_SIZES)):
            for c in range(len(CONTEXT_LENS)):
                val = grid[r, c]
                text_color = "white" if norm(val) > 0.65 else "black"
                ax.text(c, r, f"{val:.3f}", ha="center", va="center",
                        fontsize=6, color=text_color, fontweight="bold")

        ax.set_xticks(range(len(CONTEXT_LENS)))
        ax.set_xticklabels([str(c) for c in CONTEXT_LENS], fontsize=6)
        ax.set_yticks(range(len(CHUNK_SIZES)))
        ax.set_yticklabels([str(c) for c in CHUNK_SIZES], fontsize=6)

        if row_idx == 1:
            ax.set_xlabel("Cross-file Context Length", fontsize=7)
        if col_idx == 0:
            ax.set_ylabel(f"{SPLIT_LABELS[split_name]}\nChunk Size", fontsize=7)

        if row_idx == 0:
            ax.set_title(mlabel, fontweight="bold", fontsize=8, pad=4)

    # Colorbar per row
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[row_idx, :], shrink=0.8, pad=0.02, aspect=20)
    cbar.ax.tick_params(labelsize=6)
    cbar.set_label(METRIC, fontsize=7)

fig.savefig(OUTPUT_DIR / "repoeval_rq3_heatmap.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
print("Saved: output/repoeval_rq3_heatmap.pdf")

In [ ]:
# RQ3 — Line chart: parameter sensitivity (4×1 vertical stack, column-width)
# Layout: 4 rows × 1 col
#   Row 0: API-Level × Cross-file Context Length
#   Row 1: API-Level × Chunk Size
#   Row 2: Line-Level × Cross-file Context Length
#   Row 3: Line-Level × Chunk Size

METHOD_ORDER = ["function", "declaration", "sliding", "cast"]
METHOD_LABELS = ["Function", "Declaration", "Sliding Window", "cAST"]
METHOD_COLORS = ['#A0CBE8', '#4E79A7', '#F28E2B', '#E15759']
METHOD_MARKERS = ['o', 's', 'D', '^']
CHUNK_SIZES = [1000, 2000, 3000]
CONTEXT_LENS = [2048, 4096, 8192]
METRIC = "EM"

split_configs = [
    (rag[rag["split"] == "api"], "API-Level"),
    (rag[rag["split"] == "line"], "Line-Level"),
]

fig, axes = plt.subplots(4, 1, figsize=(3.5, 7), constrained_layout=True)

row = 0
for sub, task_label in split_configs:
    # Sub-row 0: X = cross-file context length
    ax = axes[row]
    for method, mlabel, color, marker in zip(METHOD_ORDER, METHOD_LABELS, METHOD_COLORS, METHOD_MARKERS):
        m = sub[sub["method"] == method]
        means, stds = [], []
        for cl in CONTEXT_LENS:
            vals = m[m["max_crossfile_context"] == cl][METRIC].values
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        ax.plot(range(len(CONTEXT_LENS)), means, color=color, marker=marker,
                markersize=5, linewidth=1.2, label=mlabel, zorder=3)
        ax.fill_between(range(len(CONTEXT_LENS)), means - stds, means + stds,
                        color=color, alpha=0.15, zorder=1)
    ax.set_xticks(range(len(CONTEXT_LENS)))
    ax.set_xticklabels([str(c) for c in CONTEXT_LENS])
    ax.set_xlabel("Cross-file Context Length", fontsize=8)
    ax.set_ylabel(METRIC, fontsize=8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, linewidth=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.set_title(task_label, fontweight="bold", fontsize=9, pad=4)

    # Sub-row 1: X = chunk size
    ax = axes[row + 1]
    for method, mlabel, color, marker in zip(METHOD_ORDER, METHOD_LABELS, METHOD_COLORS, METHOD_MARKERS):
        m = sub[sub["method"] == method]
        means, stds = [], []
        for cs in CHUNK_SIZES:
            vals = m[m["max_chunk_size"] == cs][METRIC].values
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)
        ax.plot(range(len(CHUNK_SIZES)), means, color=color, marker=marker,
                markersize=5, linewidth=1.2, label=mlabel, zorder=3)
        ax.fill_between(range(len(CHUNK_SIZES)), means - stds, means + stds,
                        color=color, alpha=0.15, zorder=1)
    ax.set_xticks(range(len(CHUNK_SIZES)))
    ax.set_xticklabels([str(c) for c in CHUNK_SIZES])
    ax.set_xlabel("Chunk Size", fontsize=8)
    ax.set_ylabel(METRIC, fontsize=8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, linewidth=0.4, zorder=0)
    ax.set_axisbelow(True)

    row += 2

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2,
           bbox_to_anchor=(0.5, -0.07), frameon=True, edgecolor="#cccccc", fontsize=8)

fig.savefig(OUTPUT_DIR / "repoeval_rq3_line.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
print("Saved: output/repoeval_rq3_line.pdf")

## RQ3: Parameter Sensitivity — Statistical Tests

Friedman test across 3 levels of each parameter (context length: 2048/4096/8192; chunk size: 1000/2000/3000),
followed by Wilcoxon signed-rank post-hoc with Holm–Bonferroni correction and Cliff's delta effect size.
Each observation = EM of one (method × retriever × LLM × other_param) configuration.

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def cliffs_delta(x, y):
    n = len(x)
    more = sum(1 for i in range(n) if x[i] > y[i])
    less = sum(1 for i in range(n) if x[i] < y[i])
    return (more - less) / n

def interpret_delta(d):
    d = abs(d)
    if d < 0.147:   return "Negligible"
    elif d < 0.33:  return "Small"
    elif d < 0.474: return "Medium"
    else:           return "Large"

CONTEXT_LENS = [2048, 4096, 8192]
CHUNK_SIZES = [1000, 2000, 3000]
METRIC = "EM"

all_results = []

for split_name in ["api", "line"]:
    sub = rag[rag["split"] == split_name]

    for param_name, param_col, levels in [
        ("Context Length", "max_crossfile_context", CONTEXT_LENS),
        ("Chunk Size", "max_chunk_size", CHUNK_SIZES),
    ]:
        # Build paired observations: each config (method × retriever × LLM × other_param) → one EM per level
        other_param = "max_chunk_size" if param_col == "max_crossfile_context" else "max_crossfile_context"
        pivot = sub.pivot_table(
            index=["method", "retriever", "llm", other_param],
            columns=param_col, values=METRIC
        )
        pivot = pivot.dropna()

        # Friedman test across 3 levels
        groups = [pivot[lv].values for lv in levels]
        fri_stat, fri_p = friedmanchisquare(*groups)

        # Post-hoc: pairwise Wilcoxon + Cliff's delta
        pairs = list(combinations(levels, 2))
        p_values = []
        posthoc = []
        for a, b in pairs:
            x, y = pivot[a].values, pivot[b].values
            stat, p = wilcoxon(x, y)
            delta = cliffs_delta(y, x)  # positive delta means b > a
            p_values.append(p)
            posthoc.append({
                "split": split_name,
                "parameter": param_name,
                "level_A": a,
                "level_B": b,
                "wilcoxon_stat": stat,
                "p_raw": p,
                "cliffs_delta": round(delta, 4),
                "effect_size": interpret_delta(delta),
            })

        rejected, corrected_p, _, _ = multipletests(p_values, method="holm")
        for i, r in enumerate(posthoc):
            r["p_corrected"] = corrected_p[i]
            r["significant"] = "Yes" if rejected[i] else "No"

        print(f"=== {split_name.upper()} — {param_name} ===")
        print(f"Friedman χ² = {fri_stat:.2f}, p = {fri_p:.2e}")
        print()

        all_results.extend(posthoc)

rq3_stat_df = pd.DataFrame(all_results)
rq3_stat_df = rq3_stat_df[["split", "parameter", "level_A", "level_B",
                            "wilcoxon_stat", "p_raw", "p_corrected",
                            "significant", "cliffs_delta", "effect_size"]]

rq3_stat_df.to_csv(OUTPUT_DIR / "repoeval_rq3_statistical_tests.csv", index=False)
print("Exported to output/repoeval_rq3_statistical_tests.csv\n")
print(rq3_stat_df.to_string(index=False))

## RQ4: Cost Trade-off — Scatter Plot

X = avg_token_cost, Y = EM.
Each point = one configuration.
Marker shape = chunking method.
Draw Pareto front connecting non-dominated points.

In [ ]:
# RQ4 — Cost-performance scatter plot with Pareto front (2×1 vertical, column-width)
# Layout: 2 rows × 1 col (top: API-Level, bottom: Line-Level)

METHOD_ORDER = ["function", "declaration", "sliding", "cast"]
METHOD_LABELS = ["Function", "Declaration", "Sliding Window", "cAST"]
METHOD_COLORS = ['#A0CBE8', '#4E79A7', '#F28E2B', '#E15759']
METHOD_MARKERS = ['o', 's', 'D', '^']
METRIC = "EM"
COST_COL = "avg_token_cost"

def pareto_front(costs, scores):
    """Return indices of Pareto-optimal points (lower cost, higher score)."""
    points = list(zip(costs, scores, range(len(costs))))
    points.sort(key=lambda x: x[0])  # sort by cost ascending
    front = []
    best_score = -np.inf
    for cost, score, idx in points:
        if score > best_score:
            front.append(idx)
            best_score = score
    return front

fig, axes = plt.subplots(2, 1, figsize=(3.5, 5), constrained_layout=True)

for row_idx, split_name in enumerate(["api", "line"]):
    ax = axes[row_idx]
    sub = rag[rag["split"] == split_name]

    agg = sub.groupby(["method", "retriever", "llm", "max_chunk_size", "max_crossfile_context"]).agg(
        em=(METRIC, "mean"),
        cost=(COST_COL, "mean"),
    ).reset_index()

    for method, mlabel, color, marker in zip(METHOD_ORDER, METHOD_LABELS, METHOD_COLORS, METHOD_MARKERS):
        m = agg[agg["method"] == method]
        ax.scatter(m["cost"], m["em"], c=color, marker=marker,
                   s=18, alpha=0.6, label=mlabel, edgecolors="none", zorder=2)

    all_costs = agg["cost"].values
    all_ems = agg["em"].values
    front_idx = pareto_front(all_costs, all_ems)
    front_costs = all_costs[front_idx]
    front_ems = all_ems[front_idx]
    sort_order = np.argsort(front_costs)
    ax.plot(front_costs[sort_order], front_ems[sort_order],
            color="black", linewidth=1.0, linestyle="--", alpha=0.7, zorder=3, label="Pareto Front")

    ax.set_xlabel("Average Token Cost", fontsize=8)
    ax.set_ylabel(METRIC, fontsize=8)
    ax.set_title({"api": "API-Level", "line": "Line-Level"}[split_name],
                 fontweight="bold", fontsize=9, pad=4)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, linewidth=0.4, zorder=0)
    ax.xaxis.grid(True, linestyle="--", alpha=0.3, linewidth=0.4, zorder=0)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.09), frameon=True, edgecolor="#cccccc", fontsize=8)

fig.savefig(OUTPUT_DIR / "repoeval_rq4_pareto.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
print("Saved: output/repoeval_rq4_pareto.pdf")

## Ablation: Sliding Window Overlap

Mean EM across overlap_lines values (0, 5, 10, 15, 20, 25), averaged over retriever × LLM × chunk_size.

In [ ]:
# Ablation: Sliding Window Overlap — Scatter + Line chart (2×1 vertical, column-width)
# Layout: 2 rows × 1 col (top: API-Level, bottom: Line-Level)
# Individual config points (scatter) + mean trend line per chunk_size

ablation_df = pd.read_csv(RESULT_DIR / "ablation_overlap.csv")

OVERLAP_VALUES = [0, 5, 10, 15, 20, 25]
CHUNK_SIZES = [1000, 2000, 3000]
CHUNK_COLORS = ['#4E79A7', '#F28E2B', '#E15759']
CHUNK_MARKERS = ['o', 's', 'D']
METRIC = "EM"

rng = np.random.default_rng(42)

fig, axes = plt.subplots(2, 1, figsize=(3.5, 4.5), constrained_layout=True)

for row_idx, split_name in enumerate(["api", "line"]):
    ax = axes[row_idx]
    sub = ablation_df[ablation_df["split"] == split_name]

    for cs, color, marker in zip(CHUNK_SIZES, CHUNK_COLORS, CHUNK_MARKERS):
        means = []
        for i, ov in enumerate(OVERLAP_VALUES):
            vals = sub[(sub["max_chunk_size"] == cs) & (sub["overlap_lines"] == ov)][METRIC].values
            means.append(np.mean(vals))
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i) + jitter, vals, c=color, marker=marker,
                       s=10, alpha=0.35, edgecolors="none", zorder=2)
        means = np.array(means)
        ax.plot(range(len(OVERLAP_VALUES)), means, color=color, marker=marker,
                markersize=5, linewidth=1.2, label=f"Chunk Size = {cs}", zorder=3)

    ax.set_xticks(range(len(OVERLAP_VALUES)))
    ax.set_xticklabels([str(v) for v in OVERLAP_VALUES])
    ax.set_xlabel("Overlap Lines", fontsize=8)
    ax.set_ylabel(METRIC, fontsize=8)
    ax.set_title({"api": "API-Level", "line": "Line-Level"}[split_name],
                 fontweight="bold", fontsize=9, pad=4)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3, linewidth=0.4, zorder=0)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.07), frameon=True, edgecolor="#cccccc", fontsize=8)

fig.savefig(OUTPUT_DIR / "repoeval_ablation_overlap.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
print("Saved: output/repoeval_ablation_overlap.pdf")

In [ ]:
# Ablation: Sliding Window Overlap — Export mean EM per (split × chunk_size × overlap_lines)

ablation_df = pd.read_csv(RESULT_DIR / "ablation_overlap.csv")

abl_summary = ablation_df.groupby(["split", "max_chunk_size", "overlap_lines"])["EM"].agg(["mean", "std", "count"]).reset_index()
abl_summary.columns = ["split", "chunk_size", "overlap_lines", "EM_mean", "EM_std", "n"]
abl_summary = abl_summary.sort_values(["split", "chunk_size", "overlap_lines"])
abl_summary["EM_mean"] = abl_summary["EM_mean"].round(4)
abl_summary["EM_std"] = abl_summary["EM_std"].round(4)

abl_summary.to_csv(OUTPUT_DIR / "repoeval_ablation_overlap_summary.csv", index=False)
print("Exported to output/repoeval_ablation_overlap_summary.csv\n")
print(abl_summary.to_string(index=False))